# Income-mixing model — feature engineering and modeling approach

This notebook builds the `income_mixing_score` model used in the analysis: a Random Forest regression that predicts how income-mixed a venue's visitors should be, given its context (borough, neighborhood income, race composition, local commercial density, subway access, and proximity to tourist/cultural draws), so the residual (actual minus expected) can flag venues that are more or less mixed than their surroundings predict.

Feature set and modeling choices:

1. **Income proxy from ACS's cumulative household-income-bracket table, not the standard median.** `median_household_income` is suppressed for roughly a fifth of NYC block groups in the standard ACS export, non-randomly — suppression is markedly more common in the Bronx than in Staten Island, and among `public_civic` places more than commercial ones. `income_proxy_v2` is instead derived by differencing ACS table A14001B's cumulative "more than $X" household-income columns into 16 discrete brackets. This is unsuppressed everywhere in NYC and correlates 0.95 with true median income where it is available (vs. 0.87 for a coarser 5-bucket proxy).
2. **`income_mixing_score` has a finite-sample bias**: within every borough its mean rises and spread shrinks as `income_known_devices` increases, so some of the most extreme scores are measurement noise from low-device venues rather than real signal (see the diagnostic cell below). Addressed with an eligibility floor (`INCOME_KNOWN_DEVICE_THRESHOLD`) and by weighting regression by device count.
3. **Random Forest, not a linear model**, because borough/category/race effects on mixing aren't additive.
4. **Race/ethnicity composition** (from a Social Explorer ACS pull, Hispanic origin by race, block-group level) is included as an independent predictor: it is not redundant with borough or income — race alone (no borough) explains more variance than income alone, and race and borough together explain more than either does alone.
5. **Gini Index of Income Inequality (ACS table A14028) is not available at block-group level** — it's 100% missing statewide, not just in NYC — so it's left out; a coarser tract-level pull would be needed to include it.
6. **Libraries are classified `public_civic`** via `sub_category == "Libraries and Archives"` rather than `top_category`, since every library branch's `top_category` is the generic "Other Information Services," shared with unrelated categories like internet publishing.
7. **Local commercial density, transit access, and proximity to tourist/cultural draws** are included as context features, each computed with a haversine BallTree over the full mapped POI universe (not just analysis-eligible venues, so density reflects real physical surroundings): POI counts within 250m/500m, distance to and count of nearby subway stops, and counts of nearby museums/historical sites/hotels (a conservative "genuinely tourist-drawing" set) and chain/franchise POIs. Tourist/cultural-draw proximity is the second most important feature in the model, after borough — several venues that look unusually well-mixed turn out to be explained by sitting near a museum or hotel rather than by anything about the venue itself. A finer venue-type covariate set (`sub_category`, chain/independent flag, footprint) and a density×borough interaction term were both tested and left out, since neither improved the model enough to justify the added complexity.

Every model variant below is evaluated with 5-fold cross-validation on the same sample, for a fair comparison across feature sets.

In [ ]:
import os
import pandas as pd
import numpy as np

# Path to the monthly analysis_pois parquet from your own Veraset/Places extract -- see README.md
DATA_DIR = os.environ.get("DATA_DIR", "./data")
poi_path = os.path.join(DATA_DIR, "analysis_pois_2026_06.parquet")
poi = pd.read_parquet(poi_path)
print(poi.shape)

## Local commercial density (computed over the *full* mapped POI universe)

Counts nearby POIs within 250m and 500m of each venue, plus a same-`top_category` count at 250m, using a haversine BallTree. This has to run on the full `poi` table (all ~91k mapped POIs, before any eligibility filter) so density reflects real physical surroundings, not just which venues happen to have enough income-known devices to be analysis-eligible.

In [ ]:
from sklearn.neighbors import BallTree

poi_coords = poi.dropna(subset=["latitude", "longitude"]).copy()
coords_rad = np.radians(poi_coords[["latitude", "longitude"]].values)
tree = BallTree(coords_rad, metric="haversine")

EARTH_R_M = 6_371_000
for radius_m in [250, 500]:
    counts = tree.query_radius(coords_rad, r=radius_m / EARTH_R_M, count_only=True)
    poi_coords[f"poi_density_{radius_m}m"] = counts - 1  # exclude self

cat = poi_coords["top_category"].values
same_cat_counts = np.zeros(len(poi_coords), dtype=int)
ind = tree.query_radius(coords_rad, r=250 / EARTH_R_M)
for i, neighbors in enumerate(ind):
    same_cat_counts[i] = np.sum(cat[neighbors] == cat[i]) - 1
poi_coords["same_category_density_250m"] = same_cat_counts

poi = poi.merge(
    poi_coords[["placekey", "poi_density_250m", "poi_density_500m", "same_category_density_250m"]],
    on="placekey", how="left"
)
poi["log_density_250m"] = np.log1p(poi["poi_density_250m"])
poi["log_density_500m"] = np.log1p(poi["poi_density_500m"])
poi["log_same_cat_density_250m"] = np.log1p(poi["same_category_density_250m"])
poi[["poi_density_250m", "poi_density_500m", "same_category_density_250m"]].describe()

## Subway accessibility

Distance to the nearest subway stop, and stop counts within 500m/800m, from MTA's own station list (`MTA_Subway_Stations.csv` — one row per stop/platform, 496 rows citywide). This is a small but independent predictor: accounting for transit access shifts several venues off the "more mixed than expected" list that turn out to be explained by proximity to a subway stop rather than anything about the venue itself.

In [ ]:
import os

SUBWAY_STATIONS_FILE = os.path.join(DATA_DIR, "MTA_Subway_Stations.csv")
assert os.path.exists(SUBWAY_STATIONS_FILE), f"missing {SUBWAY_STATIONS_FILE} -- save the MTA subway stations CSV there"

stations = pd.read_csv(SUBWAY_STATIONS_FILE).dropna(subset=["GTFS Latitude", "GTFS Longitude"])
station_rad = np.radians(stations[["GTFS Latitude", "GTFS Longitude"]].values)
subway_tree = BallTree(station_rad, metric="haversine")

poi_rad_all = np.radians(poi_coords[["latitude", "longitude"]].values)
dist, _ = subway_tree.query(poi_rad_all, k=1)
poi_coords["dist_nearest_subway_m"] = dist[:, 0] * EARTH_R_M
for radius_m, label in [(500, "500m"), (800, "800m")]:
    poi_coords[f"subway_count_{label}"] = subway_tree.query_radius(poi_rad_all, r=radius_m / EARTH_R_M, count_only=True)

poi = poi.merge(
    poi_coords[["placekey", "dist_nearest_subway_m", "subway_count_500m", "subway_count_800m"]],
    on="placekey", how="left"
)
poi["log_dist_nearest_subway"] = np.log1p(poi["dist_nearest_subway_m"])
poi["log_subway_count_500m"] = np.log1p(poi["subway_count_500m"])
poi["log_subway_count_800m"] = np.log1p(poi["subway_count_800m"])
poi[["dist_nearest_subway_m", "subway_count_500m", "subway_count_800m"]].describe()

## Proximity to tourist/cultural draws + chain density

Counts of nearby `Museums, Historical Sites, and Similar Institutions` + `Traveler Accommodation` POIs (a conservative "genuinely tourist-drawing" set, deliberately excluding the broader and noisier "Other Amusement and Recreation Industries" category) within 500m/800m, computed with the *same* BallTree already built for commercial density above. This is the second most important feature in the model, right behind borough — bigger than income, density, or transit access. A same-radius chain/franchise POI count is included too; it contributes much less (~1% of feature importance) but doesn't hurt.

In [ ]:
TOURIST_CATEGORIES = {
    "Museums, Historical Sites, and Similar Institutions",
    "Traveler Accommodation",
}
poi_coords["is_tourist_draw"] = poi_coords["top_category"].isin(TOURIST_CATEGORIES)
poi_coords["is_chain_poi"] = poi_coords["brands"].fillna("[]") != "[]"

tourist_flag = poi_coords["is_tourist_draw"].values
chain_flag = poi_coords["is_chain_poi"].values

for radius_m, label in [(500, "500m"), (800, "800m")]:
    ind = tree.query_radius(coords_rad, r=radius_m / EARTH_R_M)
    poi_coords[f"tourist_density_{label}"] = [tourist_flag[n].sum() - tourist_flag[i] for i, n in enumerate(ind)]
    poi_coords[f"chain_density_{label}"] = [chain_flag[n].sum() - chain_flag[i] for i, n in enumerate(ind)]

poi = poi.merge(
    poi_coords[["placekey", "tourist_density_500m", "tourist_density_800m", "chain_density_500m", "chain_density_800m"]],
    on="placekey", how="left"
)
for col in ["tourist_density_500m", "tourist_density_800m", "chain_density_500m", "chain_density_800m"]:
    poi[f"log_{col}"] = np.log1p(poi[col])
poi[["tourist_density_500m", "tourist_density_800m", "chain_density_500m", "chain_density_800m"]].describe()

## Encounter type classification

Venues are grouped into `public_civic`, `commercial_social`, `personal_social`, `mobility`, and `work_private` by `top_category`, with one exception: libraries are classified `public_civic` via `sub_category == "Libraries and Archives"` rather than `top_category`, since every library branch's `top_category` is the generic "Other Information Services" — shared with unrelated categories like internet publishing — rather than anything library-specific.

In [ ]:
public_civic_categories = [
    "Museums, Historical Sites, and Similar Institutions",
    "Other Amusement and Recreation Industries",
    "Civic and Social Organizations",
    "Promoters of Performing Arts, Sports, and Similar Events",
    "Colleges, Universities, and Professional Schools",
    "Performing Arts Companies",
    "Executive, Legislative, and Other General Government Support",
    "Arts, Entertainment, and Recreation",
    "Amusement Parks and Arcades",
    "Spectator Sports",
    "Religious, Grantmaking, Civic, Professional, and Similar Organizations",
]

commercial_social_categories = [
    "Restaurants and Other Eating Places",
    "Grocery Stores",
    "Specialty Food Stores",
    "Drinking Places (Alcoholic Beverages)",
    "Special Food Services",
    "Food Services and Drinking Places",
    "General Merchandise Stores, including Warehouse Clubs and Supercenters",
    "Department Stores",
    "Book Stores and News Dealers",
    "Used Merchandise Stores",
]

personal_social_categories = ["Personal Care Services", "Other Personal Services"]

mobility_categories = [
    "Transit and Ground Passenger Transportation",
    "Other Transit and Ground Passenger Transportation",
    "Taxi and Limousine Service",
]

work_private_categories = [
    "Residential Building Construction",
    "Building Equipment Contractors",
    "Offices of Real Estate Agents and Brokers",
    "Automotive Repair and Maintenance",
    "Building Finishing Contractors",
    "Other Miscellaneous Manufacturing",
    "Foundation, Structure, and Building Exterior Contractors",
    "Printing and Related Support Activities",
    "Management of Companies and Enterprises",
    "Management, Scientific, and Technical Consulting Services",
    "Architectural, Engineering, and Related Services",
    "Nonresidential Building Construction",
    "Warehousing and Storage",
]

def classify_encounter_type(row):
    cat = row["top_category"]
    if row.get("sub_category") == "Libraries and Archives":
        return "public_civic"
    elif cat in public_civic_categories:
        return "public_civic"
    elif cat in commercial_social_categories:
        return "commercial_social"
    elif cat in personal_social_categories:
        return "personal_social"
    elif cat in mobility_categories:
        return "mobility"
    elif cat in work_private_categories:
        return "work_private"
    else:
        return "other"

poi["encounter_type"] = poi.apply(classify_encounter_type, axis=1)
encounter_df = poi[poi["encounter_type"].isin(["public_civic", "commercial_social", "personal_social"])].copy()
print("libraries now included as public_civic:",
      (poi.loc[poi["sub_category"] == "Libraries and Archives", "encounter_type"] == "public_civic").all())
encounter_df.shape

## Device threshold

Venues need at least `INCOME_KNOWN_DEVICE_THRESHOLD` income-known devices to be eligible for modeling. The diagnostic cell further down shows `income_mixing_score` is noticeably biased below ~150 devices, so raising this threshold (e.g. to 100 or 150) is worth trying and re-running everything below; 50 is used here as the floor.

In [ ]:
INCOME_KNOWN_DEVICE_THRESHOLD = 50  # try 100 / 150 and compare

model_df = encounter_df[encounter_df["income_known_devices"] >= INCOME_KNOWN_DEVICE_THRESHOLD].copy()
model_df["log_devices"] = np.log1p(model_df["income_known_devices"])
model_df.shape

## Each venue's own Census block group (from its coordinates, not its visitors')

An earlier version of this pipeline assigned each venue a "neighborhood" block group by finding the most common home block group among its own visitors (the code below shows what that looked like). That's circular: it lets a venue's own visitor composition help determine the neighborhood context used to judge whether that visitor composition is surprising -- a venue's borough, local income, and race context all ended up coming from where its visitors live, not from where the venue itself sits. Checked directly against this data, the two approaches agree on the assigned block group 99.94% of the time, so the practical effect on the numbers below turns out to be small -- but the geocoded version is what's actually used going forward, since it isn't structurally circular. It spatially joins each venue's lat/long to the 2025 TIGER/Line block-group shapefile (same approach as the standalone `geocode_venues_to_cbg.py`, which this mirrors), with a nearest-polygon fallback for the handful of coastline points (piers, marinas, beaches) that don't land cleanly inside any polygon.

In [ ]:
import geopandas as gpd

# Bronx, Brooklyn (Kings), Manhattan (New York), Queens, Staten Island (Richmond)
NYC_COUNTY_FIPS = ["005", "047", "061", "081", "085"]
TIGER_BG_SHP = os.path.join(DATA_DIR, "tl_2025_36_bg", "tl_2025_36_bg.shp")

bg = gpd.read_file(TIGER_BG_SHP)
bg_nyc = bg[bg["COUNTYFP"].isin(NYC_COUNTY_FIPS)].to_crs(epsg=4326)
print(f"NYC block groups: {len(bg_nyc)}")

poi_geo = poi.dropna(subset=["latitude", "longitude"]).copy()
gdf = gpd.GeoDataFrame(
    poi_geo, geometry=gpd.points_from_xy(poi_geo["longitude"], poi_geo["latitude"]), crs="EPSG:4326"
)

joined = gpd.sjoin(gdf, bg_nyc[["GEOID", "geometry"]], how="left", predicate="within")
joined = joined.drop_duplicates(subset="placekey", keep="first")

# A handful of venues (piers, marinas, beach-adjacent points) sit exactly on a coastline
# edge and don't fall cleanly "within" any block group -- fall back to nearest polygon, in a
# projected CRS (NY State Plane, feet) so "nearest" is measured correctly.
missing_pks = joined.loc[joined["GEOID"].isna(), "placekey"]
if len(missing_pks):
    proj = gdf[gdf["placekey"].isin(missing_pks)].to_crs(epsg=2263)
    bg_nyc_proj = bg_nyc.to_crs(epsg=2263)
    nearest = gpd.sjoin_nearest(proj, bg_nyc_proj[["GEOID", "geometry"]], how="left")
    nearest = nearest.drop_duplicates(subset="placekey", keep="first")
    fallback = dict(zip(nearest["placekey"], nearest["GEOID"]))
    joined["GEOID"] = joined.apply(
        lambda r: fallback.get(r["placekey"], r["GEOID"]) if pd.isna(r["GEOID"]) else r["GEOID"], axis=1
    )
    print(f"{len(missing_pks)} venues fell back to nearest-polygon match (coastline edge cases)")

cbg_lookup = joined[["placekey", "GEOID"]].rename(columns={"GEOID": "cbg_geoid"})
cbg_lookup["cbg_geoid"] = cbg_lookup["cbg_geoid"].astype(str).str.zfill(12)

model_df = model_df.merge(cbg_lookup[["placekey", "cbg_geoid"]], on="placekey", how="left")
model_df["cbg_geoid"].isna().mean()

## Income proxy and race composition, from a Social Explorer ACS pull

`R50186212_SL150.csv` bundles three ACS tables at block-group level:

- **A14001B — Household Income, Cumulative** ("more than $10k", "more than $15k", ... "more than $200k"). Differencing consecutive columns recovers 16 discrete income brackets, unsuppressed everywhere in NYC, used to build `income_proxy_v2`.
- **A14028 — Gini Index of Income Inequality.** 100% missing at block-group level, statewide — the Census Bureau doesn't publish this statistic at that granularity. Left out entirely; a tract-level (SL140) pull would be needed, at the cost of a coarser join.
- **A04001 — Hispanic or Latino Origin by Race.** Used to build non-Hispanic White/Black/Asian/Other shares plus a Hispanic share, and a Shannon-entropy racial diversity index — an independent predictor, not redundant with borough or income.

In [ ]:
census_new = pd.read_csv(os.path.join(DATA_DIR, "acs_block_group_extract.csv"))
census_new["geoid"] = census_new["Geo__geoid_"].astype(str).str.zfill(12)

print("Gini (SE_A14028_001) NaN rate:", census_new["SE_A14028_001"].isna().mean(), "-- confirmed unusable, not used below")

# ---- income_proxy_v2 from the cumulative "more than $X" columns ----
cum_cols = [f"SE_A14001B_{i:03d}" for i in range(2, 17)]
households = census_new["SE_A14001B_001"]

under_10k = households - census_new[cum_cols[0]]
between = -census_new[cum_cols].diff(axis=1).iloc[:, 1:]   # more_than[i] - more_than[i+1] = count in that bracket
over_200k = census_new[cum_cols[-1]]

bucket_counts = pd.concat([under_10k.rename("u_10k"), between, over_200k.rename("o_200k")], axis=1)
bucket_mid = [5000, 12500, 17500, 22500, 27500, 32500, 37500, 42500, 47500, 55000, 67500, 87500, 112500, 137500, 175000, 250000]

totals = bucket_counts.sum(axis=1)
shares = bucket_counts.div(totals, axis=0)
census_new["income_proxy_v2"] = (shares.values * np.array(bucket_mid)).sum(axis=1)
census_new.loc[totals == 0, "income_proxy_v2"] = np.nan

entropy16 = -(shares * np.log(shares.where(shares > 0))).sum(axis=1)
census_new["neighborhood_income_diversity_v2"] = entropy16 / np.log(16)

# ---- race composition from Hispanic-origin-by-race ----
census_new["pop_total"] = census_new["SE_A04001_001"]
census_new["pct_hispanic"] = census_new["SE_A04001_010"] / census_new["pop_total"]
census_new["pct_white_nh"] = census_new["SE_A04001_003"] / census_new["pop_total"]
census_new["pct_black_nh"] = census_new["SE_A04001_004"] / census_new["pop_total"]
census_new["pct_asian_nh"] = census_new["SE_A04001_006"] / census_new["pop_total"]
other_nh = census_new["SE_A04001_005"] + census_new["SE_A04001_007"] + census_new["SE_A04001_008"] + census_new["SE_A04001_009"]
census_new["pct_other_nh"] = other_nh / census_new["pop_total"]

race_share_cols = ["pct_hispanic", "pct_white_nh", "pct_black_nh", "pct_asian_nh", "pct_other_nh"]
race_shares = census_new[race_share_cols]
race_entropy = -(race_shares * np.log(race_shares.where(race_shares > 0))).sum(axis=1)
census_new["racial_diversity_index"] = race_entropy / np.log(5)

census_new[["geoid", "income_proxy_v2", "neighborhood_income_diversity_v2"] + race_share_cols + ["racial_diversity_index"]].describe()

In [ ]:
model_df = model_df.merge(
    census_new[["geoid", "income_proxy_v2", "neighborhood_income_diversity_v2",
                "pct_white_nh", "pct_black_nh", "pct_hispanic", "pct_asian_nh", "pct_other_nh",
                "racial_diversity_index", "SE_A14001B_001"]].rename(columns={"SE_A14001B_001": "households"}),
    left_on="cbg_geoid", right_on="geoid", how="left"
)

borough_map = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan", "36081": "Queens", "36085": "Staten Island"}
model_df["borough"] = model_df["cbg_geoid"].str[:5].map(borough_map)

model_ready = model_df.dropna(
    subset=["income_mixing_score", "log_devices", "top_category", "income_proxy_v2"]
).copy()
model_ready["log_population"] = np.log1p(model_ready["households"])

print("model_ready shape:", model_ready.shape)
print(model_ready["borough"].value_counts())

## Local income context: spatially-smoothing `income_proxy_v2` over nearby venues

`income_proxy_v2` above comes from a single administrative census block group -- whichever one the venue's own coordinates geocode into. A real commercial corridor can straddle a block-group line, or draw its foot traffic from several adjacent block groups, none of which a venue's own single assigned block group captures on its own. This adds, for every mapped venue in the full POI universe, the average `income_proxy_v2` of all *other* mapped venues within 250m/500m (each one still using its own geographic block group) -- the same way `poi_density`/`tourist_density`/`chain_density` above already describe local context rather than relying on one administrative unit. A venue with no other income-known venue nearby falls back to its own single-block-group value rather than becoming NaN and getting silently dropped from the model.

Motivation for adding this at all: a venue's residual (`improbable_copresence`, below) correlates with the average residual of its 5-10 nearest other venues at r≈0.61 (R²≈0.37), while `top_category`/`sub_category` explain under 1% of that same variance -- hyperlocal geography is a far stronger predictor of the "surprise" than what kind of business a venue is, which is exactly what this feature is meant to capture. It turns out to be the single most important feature in the final model below, ahead of borough itself.

In [ ]:
income_by_cbg = census_new.set_index("geoid")["income_proxy_v2"]

poi_local = poi_coords.merge(cbg_lookup[["placekey", "cbg_geoid"]], on="placekey", how="left")
poi_local["income_proxy_v2"] = poi_local["cbg_geoid"].map(income_by_cbg)
income_vals = poi_local["income_proxy_v2"].to_numpy()
has_income = ~np.isnan(income_vals)

for radius_m in [250, 500]:
    ind = tree.query_radius(coords_rad, r=radius_m / EARTH_R_M)
    local_income = np.full(len(poi_local), np.nan)
    n_neighbors_known = np.zeros(len(poi_local), dtype=int)
    for i, neighbors in enumerate(ind):
        neighbors = neighbors[neighbors != i]  # exclude self
        valid = neighbors[has_income[neighbors]]
        n_neighbors_known[i] = len(valid)
        if len(valid) > 0:
            local_income[i] = income_vals[valid].mean()
    # Reliability fallback: a venue with no other income-known venue nearby (isolated
    # location, or a radius too small for this part of the city) falls back to its own
    # single-CBG value rather than becoming NaN and getting silently dropped later.
    poi_local[f"local_income_proxy_{radius_m}m"] = np.where(n_neighbors_known > 0, local_income, income_vals)
    poi_local[f"local_income_n_neighbors_{radius_m}m"] = n_neighbors_known

model_ready = model_ready.merge(
    poi_local[["placekey", "local_income_proxy_250m", "local_income_proxy_500m",
               "local_income_n_neighbors_250m", "local_income_n_neighbors_500m"]],
    on="placekey", how="left"
)
# local_income_proxy_* is a level (like income_proxy_v2 itself), not a count, so it isn't
# log-transformed -- it goes into the model on the same footing as income_proxy_v2.
print(f"median neighbors within 500m: {model_ready['local_income_n_neighbors_500m'].median():.0f}, "
      f"venues with 0 neighbors within 500m (fell back to own CBG): "
      f"{(model_ready['local_income_n_neighbors_500m'] == 0).sum()}")

## Evaluating model variants on a consistent sample, with cross-validation

Every model variant below is fit and scored on the exact same `model_ready` rows, with 5-fold CV so the reported R² isn't just a single lucky/unlucky split. Models 3 and 4 specifically test whether race is redundant with borough — it isn't.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y = model_ready["income_mixing_score"]

def evaluate(feats_num, feats_cat, label, model=None, sample_weight_col=None):
    model = model if model is not None else LinearRegression()
    X = model_ready[feats_num + feats_cat]
    pre = ColumnTransformer([
        ("num", StandardScaler(), feats_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feats_cat),
    ])
    pipe = Pipeline([("pre", pre), ("reg", model)])

    if sample_weight_col:
        w = model_ready[sample_weight_col]
        X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
            X, y, w, test_size=0.30, random_state=42
        )
        pipe.fit(X_train, y_train, reg__sample_weight=w_train)
        y_pred = pipe.predict(X_test)
        r2 = r2_score(y_test, y_pred, sample_weight=w_test)
        print(f"{label}: n={len(X)}  weighted holdout R2={r2:.4f}")
        return pipe

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    cv_r2 = cross_val_score(pipe, X, y, cv=KFold(5, shuffle=True, random_state=42), scoring="r2")
    print(f"{label}: n={len(X)}  holdout R2={r2:.4f}  MAE={mae:.4f}  |  5-fold CV R2={cv_r2.mean():.4f} (+/-{cv_r2.std():.4f})")
    return pipe

feats_num_base = ["log_devices", "income_proxy_v2", "log_population", "neighborhood_income_diversity_v2"]
feats_num_race = feats_num_base + ["pct_white_nh", "pct_black_nh", "pct_hispanic", "pct_asian_nh", "racial_diversity_index"]
feats_cat = ["top_category", "borough"]

evaluate(["log_devices"], ["top_category"], "1. devices + category")
evaluate(feats_num_base, ["top_category"], "2. + income_proxy_v2 + population + diversity")
evaluate(feats_num_base, feats_cat, "3. + borough (linear)")
evaluate(feats_num_race, ["top_category"], "4. + race, NO borough")
model_linear = evaluate(feats_num_race, feats_cat, "5. + race + borough (linear)")

## Non-linear models

A Random Forest / Gradient Boosting model on the exact same features scores meaningfully higher than the linear model, evidence the relationships are non-additive (e.g. category effects differ by borough) rather than evidence of a missing feature.

From here on, models also include local commercial density, subway accessibility, and tourist/chain density (`log_density_250m` / `log_density_500m` / `log_same_cat_density_250m` / `log_dist_nearest_subway` / `log_subway_count_500m` / `log_subway_count_800m` / `log_tourist_density_500m` / `log_tourist_density_800m` / `log_chain_density_500m` / `log_chain_density_800m`). A finer venue-level covariate set (`sub_category`, chain/independent flag, footprint) was tested separately and left out — it changed zero venues on the outlier lists and moved CV R² by less than the fold-to-fold noise, so venue *type* isn't a missing confound here. Tourist-draw proximity, density, and subway access are different: each is included below because it materially changes both model fit and the outlier lists. (A density×borough interaction was also tested — density's effect on mixing does reverse sign in the Bronx relative to every other borough, a real and highly significant pattern — but adding it as an explicit feature destabilized the forest, which can already represent that interaction through ordinary splits; kept as an interpretive finding, not a model feature.)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

feats_num_full = feats_num_race + [
    "log_density_250m", "log_density_500m", "log_same_cat_density_250m",
    "log_dist_nearest_subway", "log_subway_count_500m", "log_subway_count_800m",
    "log_tourist_density_500m", "log_tourist_density_800m",
    "log_chain_density_500m", "log_chain_density_800m",
    "local_income_proxy_250m", "local_income_proxy_500m",
]

model_rf = evaluate(feats_num_full, feats_cat, "6. RandomForest (income_v2 + race + borough + density + transit + tourist/chain)",
                     RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))
model_gbm = evaluate(feats_num_full, feats_cat, "7. GradientBoosting",
                      GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42))

In [ ]:
# Feature importance from the Random Forest — race should show up as comparable to income, not swamped by borough
names = model_rf.named_steps["pre"].get_feature_names_out()
importances = model_rf.named_steps["reg"].feature_importances_
imp_df = pd.DataFrame({"feature": names, "importance": importances}).sort_values("importance", ascending=False)
imp_df.head(15)

## Diagnostic: is `income_mixing_score` biased at low device counts?

This checks, within each borough separately, whether the mean and spread of the target shift as `income_known_devices` grows. If they do — and they do, in every borough — that's a small-sample bias in the entropy-style score itself, not a modeling problem you can fix with better features. It argues for either raising `INCOME_KNOWN_DEVICE_THRESHOLD` above, or weighting regression by device count (next cell).

In [ ]:
model_ready["device_bucket"] = pd.cut(
    model_ready["income_known_devices"], [0, 75, 150, 300, 1_000_000],
    labels=["50-75", "75-150", "150-300", "300+"]
)
model_ready.groupby(["borough", "device_bucket"], observed=True)["income_mixing_score"].agg(["mean", "std", "count"])

## Weighting regression by device count, to down-weight noisy, low-device venues

Tested here on a linear-regression variant for a clean before/after comparison. The final RandomForest model and the outlier-flagging table further below now carry this same device-count weighting through as well -- it had previously been demonstrated here but never actually applied to the model used for the headline residual/outlier results; that gap is fixed starting in the section below.

In [ ]:
model_weighted = evaluate(
    feats_num_full, feats_cat, "8. Weighted linear regression (income_v2 + race + borough + density, by income_known_devices)",
    sample_weight_col="income_known_devices"
)

## Residual/outlier table (`improbable_copresence`)

Every row's prediction comes from a fold that never saw that row's own label. CV here is grid-cell-blocked (`GroupKFold`, venues bucketed into ~2000ft/610m cells projected to NY State Plane feet, so a venue's own physical neighbors can't leak into its training fold) rather than a random row-level split -- nearby venues' residuals are correlated (see the spatial-autocorrelation diagnostic later in this notebook and README's Known limitations, r≈0.52-0.61), so a random split inflates apparent CV R² relative to genuine out-of-sample generalization to an unseen neighborhood. The original random-row `KFold(5, shuffle=True, random_state=42)` is still fit alongside it and printed below as a secondary comparison number only -- it doesn't feed anything after this cell. Both are looped manually (rather than via `cross_val_predict`) so each fold's `sample_weight` can be sliced the same way `X`/`y` are, using the full income/race/borough/density/transit/tourist/local-income feature set and the RandomForest model, since it scores highest in CV above. The fit is weighted by `income_known_devices`, closing the gap noted above -- this is the first place in the notebook where that weighting actually reaches the model used for the headline results, not just a side comparison.

**Interpretive note:** with race baked into what "expected" mixing looks like for a neighborhood, these residuals answer "more/less mixed than typical *given* local income, category, race, and borough" -- not an absolute standard. Worth stating explicitly in any write-up, since "expected" here is doing some normative work: it takes current racial-composition patterns as the baseline to measure deviation from.


In [ ]:
from sklearn.model_selection import GroupKFold

sample_weight_full = model_ready["income_known_devices"].values

X_full = model_ready[feats_num_full + feats_cat]
pre = ColumnTransformer([
    ("num", StandardScaler(), feats_num_full),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feats_cat),
])

# Grid-cell spatial blocking: project to NY State Plane feet, bucket into ~2000ft cells, so
# nearby venues (whose residuals are correlated -- see the spatial-autocorrelation diagnostic
# later in this notebook) land in the same fold instead of splitting across train/test.
GRID_CELL_FT = 2000
_gdf = gpd.GeoDataFrame(
    model_ready, geometry=gpd.points_from_xy(model_ready["longitude"], model_ready["latitude"]), crs="EPSG:4326"
).to_crs(epsg=2263)
_gx = (_gdf.geometry.x // GRID_CELL_FT).astype(int).astype(str)
_gy = (_gdf.geometry.y // GRID_CELL_FT).astype(int).astype(str)
grid_cell = (_gx + "_" + _gy).values
print(f"grid cells: {len(np.unique(grid_cell))} distinct {GRID_CELL_FT}ft cells for {len(model_ready)} venues")

# Fit manually per fold (rather than via cross_val_predict) so sample_weight can be sliced
# the same way X and y are for each fold -- cross_val_predict doesn't support that directly.
#
# PRIMARY: grid-cell-blocked GroupKFold. Feeds expected_mixing_cv / residuals / z-scores /
# flagging in the cells below.
gkf_main = GroupKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = np.full(len(model_ready), np.nan)
fold_r2 = []
for tr_idx, te_idx in gkf_main.split(X_full, y, groups=grid_cell):
    fold_pipe = Pipeline([("pre", pre), ("reg", RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))])
    fold_pipe.fit(X_full.iloc[tr_idx], y.iloc[tr_idx], reg__sample_weight=sample_weight_full[tr_idx])
    pred = fold_pipe.predict(X_full.iloc[te_idx])
    y_pred_cv[te_idx] = pred
    fold_r2.append(r2_score(y.iloc[te_idx], pred, sample_weight=sample_weight_full[te_idx]))
cv_full = np.array(fold_r2)
print(f"PRIMARY (grid-cell GroupKFold) weighted CV R2={cv_full.mean():.4f} (+/-{cv_full.std():.4f})")

# SECONDARY, reported for comparison only -- NOT used for residuals/z-scores/flagging below.
# This is the original random row-level KFold; it can overstate out-of-sample R^2 because
# spatially/residually-correlated neighbor venues can split across its train/test folds.
kf_random = KFold(5, shuffle=True, random_state=42)
fold_r2_random = []
for tr_idx, te_idx in kf_random.split(X_full):
    fold_pipe = Pipeline([("pre", pre), ("reg", RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))])
    fold_pipe.fit(X_full.iloc[tr_idx], y.iloc[tr_idx], reg__sample_weight=sample_weight_full[tr_idx])
    pred = fold_pipe.predict(X_full.iloc[te_idx])
    fold_r2_random.append(r2_score(y.iloc[te_idx], pred, sample_weight=sample_weight_full[te_idx]))
cv_random = np.array(fold_r2_random)
print(f"SECONDARY (random row-level KFold, comparison only -- not used below) weighted CV R2={cv_random.mean():.4f} (+/-{cv_random.std():.4f})")

rf_pipe = Pipeline([("pre", pre), ("reg", RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))])
rf_pipe.fit(X_full, y, reg__sample_weight=sample_weight_full)

names = rf_pipe.named_steps["pre"].get_feature_names_out()
imps = rf_pipe.named_steps["reg"].feature_importances_
imp_df = pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False)
print("\ntop 15 feature importances (final, weighted, geocoded-CBG model -- supersedes the")
print("exploratory unweighted ranking earlier in the notebook):")
print(imp_df.head(15).to_string(index=False))

results_cv = model_ready.copy()
results_cv["expected_mixing_cv"] = y_pred_cv
results_cv["improbable_copresence_cv"] = results_cv["income_mixing_score"] - results_cv["expected_mixing_cv"]

results_cv.groupby("borough")["improbable_copresence_cv"].agg(["count", "mean", "std", "min", "max"])


## Why raw residuals aren't enough to rank "surprising" venues: borough-normalize them

The Bronx's residuals have a much wider spread than the other boroughs (see the table above). That means a raw, city-wide "top 30 most-mixed-than-expected" ranking will be dominated by whichever borough has the widest spread, regardless of whether it actually has more genuinely surprising venues. Checked directly on this data (June 2026, weighted, geocoded-CBG pipeline, full feature set, grid-cell-blocked CV -- see above): the raw top-30 by `improbable_copresence_cv` is **29/30 Bronx** (1 Staten Island) -- almost every other borough is shut out entirely. The raw bottom-30 is more spread out but still Bronx-heavy (14 Bronx, 10 Queens, 3 Staten Island, 2 Brooklyn, 1 Manhattan). Once you z-score residuals *within* each borough (comparing a venue only to others in the same borough), the top-30 spreads out (Bronx 13, Staten Island 7, Manhattan 5, Queens 4, Brooklyn 1) and the bottom-30 shifts to Brooklyn (19), Queens (10), Manhattan (1) -- a near-total reversal from the raw ranking, driven largely by variance differences between boroughs, not by where the "real" outliers are. Use the z-scored version for any city-wide ranking or map; the raw version is only meaningful for comparing venues within the same borough.

Z-scoring isn't the whole fix, though: the ranking above still uses a flat top/bottom-30 *count*, which doesn't account for boroughs having very different numbers of eligible venues in the first place -- see the flagging rule used below.


In [ ]:
results_cv["improbable_copresence_z"] = (
    results_cv.groupby("borough")["improbable_copresence_cv"]
    .transform(lambda s: (s - s.mean()) / s.std())
)

cols = ["location_name", "borough", "top_category", "income_known_devices",
        "income_mixing_score", "expected_mixing_cv", "improbable_copresence_cv", "improbable_copresence_z"]

print("Raw top-30 borough counts:", results_cv.nlargest(30, "improbable_copresence_cv")["borough"].value_counts().to_dict())
print("Raw bottom-30 borough counts:", results_cv.nsmallest(30, "improbable_copresence_cv")["borough"].value_counts().to_dict())
print("Z-scored top-30 borough counts:", results_cv.nlargest(30, "improbable_copresence_z")["borough"].value_counts().to_dict())
print("Z-scored bottom-30 borough counts:", results_cv.nsmallest(30, "improbable_copresence_z")["borough"].value_counts().to_dict())

results_cv.nlargest(15, "improbable_copresence_z")[cols]

## Outlier flagging and bootstrap stability check: which "surprising" venues are real vs. noise?

Even after z-scoring within borough, the ranking above still uses a flat top/bottom-30 *count*, picked without regard to borough size -- and boroughs aren't remotely the same size (Staten Island has a small fraction of Brooklyn's venue count). A flat count over- or under-represents boroughs relative to how many eligible venues they actually have, and is sensitive to noise right at the cutoff: two of this project's flagship venues (Jacob Riis Beach, Chens Fresh Food Market) were found to be genuinely extreme for their own borough but narrowly missed a flat citywide top/bottom-30 cutoff in some months -- exactly the failure mode a fixed-count rule is prone to. The fix used below instead flags the top/bottom `PCT`% of venues **within each borough separately** (0.5% here), so every borough contributes outliers in proportion to its own size.

Across ~28,000 venues, the tails of any residual distribution contain some pure noise -- a venue can land in the flagged set once just from where the random splits happened to fall. This refits the Random Forest on `N_BOOT` bootstrap resamples (random resamples of the same rows, with replacement, weighted by device count same as the reference fit), **re-does the borough z-scoring and the borough-proportional flagging on each resample independently** (not just once on the reference fit -- otherwise this would only test the raw residual's stability, not the flagging rule actually used), and for each venue in the reference flagged set, records what fraction of resamples *also* flag it under the resample's own ranking. A venue near 100% is a stable finding; a venue under 50% landed there more by chance of this particular fit than a robust signal.

Run on the real June 2026 data behind this notebook (`N_BOOT=30`, weighted, geocoded-CBG pipeline, full feature set including local income context, `PCT=0.5`): PRIMARY (grid-cell-blocked) CV R²=0.7197 (matching the standalone `fit_mixing_model_2026_06.py` exactly). The original random-row CV R²=0.7574 is still computed above as a secondary comparison figure, but no longer feeds anything below -- random row-level folds let spatially-correlated neighbor venues leak between train and test, inflating that number (see the spatial-blocking note earlier in this notebook and README's Known limitations). The borough-proportional rule flags 141 venues on each side (Brooklyn 45, Queens 37, Manhattan 32, Bronx 18, Staten Island 9 -- proportional to each borough's share of eligible venues; these counts are fixed by borough size and `PCT`, so they're unchanged by the CV fix). Re-run under the corrected, spatially-blocked residuals, the **bottom 141** ("less mixing than expected") is still notably more stable than the top 141, though both dropped somewhat from the earlier random-fold numbers: bottom 104/141 at ≥80% stability (mean 81.0%, 22/141 below 50%, down from mean 88.7% under the old CV); top 60/141 at ≥80% stability (mean 56.7%, 64/141 below 50%, down from mean 68.9%). Don't treat the "more mixed" list as equally trustworthy as the "less mixed" list without noting that gap -- if anything it's a bit wider now than it looked under the leakier validation.

The geographic patterns that motivated switching to this rule hold up cleanly under the corrected CV. On the "more mixed than expected" side, all three Jacob Riis fragments clear the bootstrap bar at 100% stability in the same month: Jacob Riis Beach, The People's Beach at Jacob Riis Park, and Jacob Riis Park, alongside Hunts Point Floating Pier (Bronx, 100%) and Amenazel Organic (Bronx, 96.7%) -- identical stability figures to the old random-fold CV, for what it's worth, even though the aggregate top-141 mean dropped: these specific venues are extreme enough to be robust either way. On the "less mixed than expected" side, Chens Fresh Food Market (100%) anchors a larger, still mostly Brooklyn-based cluster of otherwise-unrelated small businesses. Lower `N_BOOT` below if this is too slow on your machine -- even 10-15 resamples gives a useful signal, just noisier.


In [ ]:
PCT = 0.5  # top/bottom 0.5% of venues, computed separately within each borough
N_BOOT = 30  # lower this (e.g. 10-15) if it's too slow

def borough_proportional_flags(z, borough, pct):
    """Returns (top_mask, bottom_mask): top/bottom pct% of venues, computed WITHIN each borough."""
    top_mask = np.zeros(len(z), dtype=bool)
    bottom_mask = np.zeros(len(z), dtype=bool)
    z = np.asarray(z)
    borough = np.asarray(borough)
    for bor in np.unique(borough):
        idx = np.where(borough == bor)[0]
        k = max(1, int(round(len(idx) * pct / 100.0)))
        order = np.argsort(z[idx])
        bottom_mask[idx[order[:k]]] = True
        top_mask[idx[order[-k:]]] = True
    return top_mask, bottom_mask

top_mask_ref, bottom_mask_ref = borough_proportional_flags(
    results_cv["improbable_copresence_z"].values, results_cv["borough"].values, PCT
)
top_ref = results_cv.loc[top_mask_ref, "placekey"].tolist()
bottom_ref = results_cv.loc[bottom_mask_ref, "placekey"].tolist()
print(f"borough-proportional flags: {len(top_ref)} top, {len(bottom_ref)} bottom")
print("top borough counts:", results_cv.loc[top_mask_ref, "borough"].value_counts().to_dict())
print("bottom borough counts:", results_cv.loc[bottom_mask_ref, "borough"].value_counts().to_dict())

top_counts = {pk: 0 for pk in top_ref}
bottom_counts = {pk: 0 for pk in bottom_ref}

placekeys = model_ready["placekey"].values
borough_arr = model_ready["borough"].values
n = len(model_ready)
rng = np.random.RandomState(0)

for b in range(N_BOOT):
    idx = rng.randint(0, n, size=n)
    Xb, yb = X_full.iloc[idx], y.iloc[idx]
    boot_pipe = Pipeline([("pre", pre), ("reg", RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=20, random_state=b, n_jobs=-1))])
    boot_pipe.fit(Xb, yb, reg__sample_weight=sample_weight_full[idx])
    resid = y.values - boot_pipe.predict(X_full)

    # borough-renormalize THIS resample's residuals, same as the reference ranking does --
    # otherwise this only tests raw-residual stability, not the flagging rule actually used.
    resid_z = np.empty_like(resid)
    for bor in np.unique(borough_arr):
        mask = borough_arr == bor
        r = resid[mask]
        resid_z[mask] = (r - r.mean()) / r.std()

    top_mask_b, bottom_mask_b = borough_proportional_flags(resid_z, borough_arr, PCT)
    top_pks_b, bottom_pks_b = set(placekeys[top_mask_b]), set(placekeys[bottom_mask_b])
    for pk in top_ref:
        if pk in top_pks_b:
            top_counts[pk] += 1
    for pk in bottom_ref:
        if pk in bottom_pks_b:
            bottom_counts[pk] += 1

results_cv["stability_pct"] = np.nan
for pk, c in {**top_counts, **bottom_counts}.items():
    results_cv.loc[results_cv["placekey"] == pk, "stability_pct"] = c / N_BOOT * 100

stable_top = results_cv[results_cv["placekey"].isin(top_ref)].sort_values("improbable_copresence_z", ascending=False)
stable_bottom = results_cv[results_cv["placekey"].isin(bottom_ref)].sort_values("improbable_copresence_z")

n_top, n_bottom = len(stable_top), len(stable_bottom)
print(f"Top ({n_top} venues): {(stable_top['stability_pct'] >= 80).sum()}/{n_top} at >=80% stability, "
      f"{(stable_top['stability_pct'] < 50).sum()}/{n_top} below 50%, mean={stable_top['stability_pct'].mean():.1f}%")
print(f"Bottom ({n_bottom} venues): {(stable_bottom['stability_pct'] >= 80).sum()}/{n_bottom} at >=80% stability, "
      f"{(stable_bottom['stability_pct'] < 50).sum()}/{n_bottom} below 50%, mean={stable_bottom['stability_pct'].mean():.1f}%")

stable_top[cols + ["stability_pct"]]